In [1]:
import torch


def load_yolo_model(path):
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    model = (ckpt.get("ema") or ckpt["model"]).float().eval()
    return ckpt, model


yolov8_ckpt, yolov8 = load_yolo_model("./yolov8n.pt")
yolov_vif_ckpt, yolov_vif = load_yolo_model("./yolov8_vif.pt")

print(type(yolov8), type(yolov_vif))
yolov8_dict = yolov8.state_dict()
yolov_vif_dict = yolov_vif.state_dict()

yolov8_keys = set(yolov8_dict.keys())
yolov_vif_keys = set(yolov_vif_dict.keys())
only_in_yolov8 = sorted(yolov8_keys - yolov_vif_keys)
only_in_yolov_vif = sorted(yolov_vif_keys - yolov8_keys)
common_keys = sorted(yolov8_keys & yolov_vif_keys)

print(f"yolov8 key count: {len(yolov8_keys)}")
print(f"yolov_vif key count: {len(yolov_vif_keys)}")
print(f"common key count: {len(common_keys)}")
print(f"only in yolov8: {len(only_in_yolov8)}")
print(f"only in yolov_vif: {len(only_in_yolov_vif)}")

print("\nKeys only in yolov8:")
print(only_in_yolov8)

print("\nKeys only in yolov_vif:")
print(only_in_yolov_vif)


<class 'ultralytics.nn.tasks.DetectionModel'> <class 'ultralytics.nn.tasks.DetectionModel'>
yolov8 key count: 355
yolov_vif key count: 411
common key count: 162
only in yolov8: 193
only in yolov_vif: 249

Keys only in yolov8:
['model.12.cv1.bn.bias', 'model.12.cv1.bn.num_batches_tracked', 'model.12.cv1.bn.running_mean', 'model.12.cv1.bn.running_var', 'model.12.cv1.bn.weight', 'model.12.cv1.conv.weight', 'model.12.cv2.bn.bias', 'model.12.cv2.bn.num_batches_tracked', 'model.12.cv2.bn.running_mean', 'model.12.cv2.bn.running_var', 'model.12.cv2.bn.weight', 'model.12.cv2.conv.weight', 'model.12.m.0.cv1.bn.bias', 'model.12.m.0.cv1.bn.num_batches_tracked', 'model.12.m.0.cv1.bn.running_mean', 'model.12.m.0.cv1.bn.running_var', 'model.12.m.0.cv1.bn.weight', 'model.12.m.0.cv1.conv.weight', 'model.12.m.0.cv2.bn.bias', 'model.12.m.0.cv2.bn.num_batches_tracked', 'model.12.m.0.cv2.bn.running_mean', 'model.12.m.0.cv2.bn.running_var', 'model.12.m.0.cv2.bn.weight', 'model.12.m.0.cv2.conv.weight', 'mode

In [2]:
import copy
import re
from collections import OrderedDict

# 按当前 state_dict 的实际编号，yolov_vif 额外插入的是 model.10.*。
# 因此前面的 model.0-model.9 直接取 yolov8，model.10 保留 yolov_vif，后续层再把 yolov8 的层号整体加 1 对齐。
layer_pattern = re.compile(r"^model\.(\d+)\.(.+)$")
inserted_layer_idx = 10


def build_mixed_yolov_vif_dict(yolov8_dict, yolov_vif_dict, inserted_layer_idx=10):
    mixed_dict = OrderedDict()
    copied_from_yolov8 = 0
    kept_from_yolov_vif = 0
    missing_source_keys = []
    shape_mismatches = []

    for vif_key, vif_value in yolov_vif_dict.items():
        match = layer_pattern.match(vif_key)
        if not match:
            mixed_dict[vif_key] = vif_value.clone()
            kept_from_yolov_vif += 1
            continue

        vif_layer_idx = int(match.group(1))
        suffix = match.group(2)

        if vif_layer_idx < inserted_layer_idx:
            source_key = vif_key
        elif vif_layer_idx == inserted_layer_idx:
            source_key = None
        else:
            source_key = f"model.{vif_layer_idx - 1}.{suffix}"

        if source_key is None:
            mixed_dict[vif_key] = vif_value.clone()
            kept_from_yolov_vif += 1
            continue

        source_value = yolov8_dict.get(source_key)
        if source_value is None:
            mixed_dict[vif_key] = vif_value.clone()
            kept_from_yolov_vif += 1
            missing_source_keys.append((vif_key, source_key))
            continue

        if source_value.shape != vif_value.shape:
            mixed_dict[vif_key] = vif_value.clone()
            kept_from_yolov_vif += 1
            shape_mismatches.append((vif_key, source_key, tuple(vif_value.shape), tuple(source_value.shape)))
            continue

        mixed_dict[vif_key] = source_value.clone()
        copied_from_yolov8 += 1

    return mixed_dict, copied_from_yolov8, kept_from_yolov_vif, missing_source_keys, shape_mismatches


mixed_yolov_vif_dict, copied_from_yolov8, kept_from_yolov_vif, missing_source_keys, shape_mismatches = build_mixed_yolov_vif_dict(
    yolov8_dict,
    yolov_vif_dict,
    inserted_layer_idx=inserted_layer_idx,
)

hybrid_yolov_vif = copy.deepcopy(yolov_vif)
load_result = hybrid_yolov_vif.load_state_dict(mixed_yolov_vif_dict, strict=True)

print(f"mixed key count: {len(mixed_yolov_vif_dict)}")
print(f"copied from yolov8: {copied_from_yolov8}")
print(f"kept from yolov_vif: {kept_from_yolov_vif}")
print(f"missing source keys: {len(missing_source_keys)}")
print(f"shape mismatches: {len(shape_mismatches)}")
print(load_result)
print(list(mixed_yolov_vif_dict.keys()))

if missing_source_keys:
    print("\nMissing source key samples:")
    print(missing_source_keys[:10])

if shape_mismatches:
    print("\nShape mismatch samples:")
    print(shape_mismatches[:10])

mixed_ckpt = dict(yolov_vif_ckpt)
mixed_ckpt["model"] = copy.deepcopy(hybrid_yolov_vif).float()
if "ema" in mixed_ckpt:
    mixed_ckpt["ema"] = None

torch.save(mixed_ckpt, "mixed_yolov_vif.pt")

mixed key count: 411
copied from yolov8: 319
kept from yolov_vif: 92
missing source keys: 0
shape mismatches: 36
<All keys matched successfully>
['model.0.conv.weight', 'model.0.bn.weight', 'model.0.bn.bias', 'model.0.bn.running_mean', 'model.0.bn.running_var', 'model.0.bn.num_batches_tracked', 'model.1.conv.weight', 'model.1.bn.weight', 'model.1.bn.bias', 'model.1.bn.running_mean', 'model.1.bn.running_var', 'model.1.bn.num_batches_tracked', 'model.2.cv1.conv.weight', 'model.2.cv1.bn.weight', 'model.2.cv1.bn.bias', 'model.2.cv1.bn.running_mean', 'model.2.cv1.bn.running_var', 'model.2.cv1.bn.num_batches_tracked', 'model.2.cv2.conv.weight', 'model.2.cv2.bn.weight', 'model.2.cv2.bn.bias', 'model.2.cv2.bn.running_mean', 'model.2.cv2.bn.running_var', 'model.2.cv2.bn.num_batches_tracked', 'model.2.m.0.cv1.conv.weight', 'model.2.m.0.cv1.bn.weight', 'model.2.m.0.cv1.bn.bias', 'model.2.m.0.cv1.bn.running_mean', 'model.2.m.0.cv1.bn.running_var', 'model.2.m.0.cv1.bn.num_batches_tracked', 'model.2